# EquiFrame Comparación nativa de tres motores con 10000 registros

Este cuaderno ejecuta una agregación de 10.000 transacciones reales mediante
Pandas, Polars y DuckDB. Comprueba sus resultados contra un cálculo separado,
contrasta las tres parejas y extrae perfiles agrupados 1.1. También comprueba
un cambio controlado que mantiene el total global.

En Colab usa CPU y **Ejecutar todas** en orden. No necesita GPU, cuentas Kaggle
ni credenciales del proyecto. Cada motor recibe una lectura tipada de un
formato distinto; su transformación se ejecuta nativamente.


In [1]:
import subprocess
import sys

# Archivo público inmutable de la versión auditada. No requiere fichas API.
wheel = (
    "https://github.com/htroya/equiframe/releases/download/v0.14.0/"
    "equiframe-0.14.0-py3-none-any.whl"
    "#sha256=c1956d8cfeca3b558fccda9955d1cdba235934041df67ed753cca698ed461219"
)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "equiframe[engines] @ " + wheel,
    "pandas==3.1.0", "polars==1.44.2", "duckdb==1.5.6", "pyarrow==25.0.1",
])
subprocess.check_call([sys.executable, "-m", "pip", "check"])
print("Instalación de EquiFrame 0.14.0 comprobada.")


Instalación de EquiFrame 0.14.0 comprobada.


## Fuente y selección de los datos

Se descargan datos reales de **E-Commerce Data**, versión 1, desde
[Kaggle](https://www.kaggle.com/datasets/carrie1/ecommerce-data).
Su fuente original es **Online Retail**, de Daqing Chen, publicada por UCI:
[https://doi.org/10.24432/C5BW33](https://doi.org/10.24432/C5BW33).
UCI declara la licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Esta atribución se conserva; no se atribuye a Kaggle una licencia CC0.

Se seleccionan las **primeras 10.000 líneas de transacción**, sin replicar registros,
eliminar devoluciones ni suprimir clientes ausentes. Es una selección didáctica
determinista, no una muestra representativa para inferencia estadística.
La descarga pública no usa cuentas, contraseñas ni claves API.

La clave técnica `row_id` identifica la posición de cada registro seleccionado;
no sustituye un identificador de negocio. Los identificadores originales se
conservan como texto. Los campos vacíos de descripción y cliente se interpretan
como ausencias conforme a una convención explícita de lectura. La fecha se
interpreta como mes/día/año, sin inventar una zona horaria. Los precios se
conservan como `Decimal`; `amount_minor` expresa el importe firmado en peniques
de GBP y mantiene las cantidades negativas de los registros originales.


In [2]:
import csv
import hashlib
import io
import itertools
import json
import tempfile
import zipfile
from datetime import datetime
from decimal import Decimal
from pathlib import Path
from urllib.request import Request, urlopen

import equiframe as ef
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

N = 10_000
source_url = (
    "https://www.kaggle.com/api/v1/datasets/download/carrie1/ecommerce-data"
    "?datasetVersionNumber=1"
)
with urlopen(Request(source_url, headers={"User-Agent": "EquiFrame-public-tutorial/1.0"}),
             timeout=60) as response:
    payload = response.read(60_000_001)
if len(payload) > 60_000_000:
    raise RuntimeError("La descarga excede el límite declarado.")
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    if archive.namelist() != ["data.csv"]:
        raise RuntimeError("El inventario de la fuente cambió.")
    csv_bytes = archive.read("data.csv")
source_sha256 = hashlib.sha256(csv_bytes).hexdigest()
if source_sha256 != "d07aec9960083af2339975a3f9d3b26313b342dcd9f86cce0b919b1cde639a44":
    raise RuntimeError("La fuente no coincide con la versión examinada.")
reader = csv.DictReader(io.StringIO(csv_bytes.decode("ISO-8859-1"), newline=""))
original_rows = list(itertools.islice(reader, N))
source_rows = N + sum(1 for _ in reader)
if len(original_rows) != N or source_rows != 541_909:
    raise RuntimeError("El tamaño de la fuente o de la selección cambió.")

records = []
for row_id, row in enumerate(original_rows, 1):
    price = Decimal(row["UnitPrice"])
    quantity = int(row["Quantity"])
    minor = price * quantity * 100
    if price.as_tuple().exponent < -2 or minor != minor.to_integral_value():
        raise RuntimeError("El precio requiere otra escala monetaria.")
    records.append({
        "row_id": row_id, "invoice_no": row["InvoiceNo"],
        "stock_code": row["StockCode"], "description": row["Description"] or None,
        "quantity": quantity,
        "invoice_date": datetime.strptime(row["InvoiceDate"], "%m/%d/%Y %H:%M"),
        "unit_price": price, "customer_id": row["CustomerID"] or None,
        "country": row["Country"], "currency": "GBP", "amount_minor": int(minor),
    })
schema = pa.schema([
    ("row_id", pa.int64()), ("invoice_no", pa.string()), ("stock_code", pa.string()),
    ("description", pa.string()), ("quantity", pa.int64()),
    ("invoice_date", pa.timestamp("us")), ("unit_price", pa.decimal128(18, 2)),
    ("customer_id", pa.string()), ("country", pa.string()),
    ("currency", pa.string()), ("amount_minor", pa.int64()),
])
original_table = pa.Table.from_pylist(records, schema=schema)
temporary = tempfile.TemporaryDirectory(prefix="equiframe-kaggle-")
work = Path(temporary.name)
print(json.dumps({
    "version_equiframe": ef.__version__, "filas_fuente": source_rows,
    "filas_seleccionadas": len(records), "sha256_csv_original": source_sha256,
    "clientes_ausentes": sum(r["customer_id"] is None for r in records),
    "descripciones_ausentes": sum(r["description"] is None for r in records),
    "cantidades_negativas": sum(r["quantity"] < 0 for r in records),
}, ensure_ascii=False, indent=2))


{
  "version_equiframe": "0.14.0",
  "filas_fuente": 541909,
  "filas_seleccionadas": 10000,
  "sha256_csv_original": "d07aec9960083af2339975a3f9d3b26313b342dcd9f86cce0b919b1cde639a44",
  "clientes_ausentes": 2291,
  "descripciones_ausentes": 42,
  "cantidades_negativas": 130
}


## Tres formatos con una especificación de lectura común

La lectura y la ejecución cumplen funciones distintas. Se verifica primero
que CSV, JSONL y Parquet conservan los mismos registros. Después cada motor
realiza su propia agregación; convertir una tabla no se presenta como una
ejecución independiente del ETL.


In [3]:
# CSV y JSONL utilizan representaciones textuales explícitas para Decimal y fechas.
# Parquet conserva sus tipos. La misma especificación reconstruye las tres entradas.
serial = [dict(r, invoice_date=r["invoice_date"].isoformat(),
               unit_price=format(r["unit_price"], ".2f")) for r in records]
csv_path, jsonl_path, parquet_path = work / "retail.csv", work / "retail.jsonl", work / "retail.parquet"
with csv_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=schema.names)
    writer.writeheader()
    writer.writerows(serial)
with jsonl_path.open("w", encoding="utf-8") as stream:
    for row in serial:
        stream.write(json.dumps(row, ensure_ascii=False) + "\n")
pq.write_table(original_table, parquet_path)

def typed(rows):
    restored = []
    for item in rows:
        row = dict(item)
        for name in ("row_id", "quantity", "amount_minor"):
            row[name] = int(row[name])
        row["invoice_date"] = datetime.fromisoformat(row["invoice_date"])
        row["unit_price"] = Decimal(row["unit_price"])
        for name in ("description", "customer_id"):
            if row[name] == "":
                row[name] = None
        restored.append(row)
    return pa.Table.from_pylist(restored, schema=schema)

with csv_path.open(encoding="utf-8", newline="") as stream:
    csv_table = typed(csv.DictReader(stream))
with jsonl_path.open(encoding="utf-8") as stream:
    jsonl_table = typed(json.loads(line) for line in stream)
parquet_table = pq.read_table(parquet_path)
sources = {"CSV": csv_table, "JSONL": jsonl_table, "Parquet": parquet_table}
policy = ef.ComparisonPolicy(
    index="ignore", physical_types="lossless",
    justification="Las columnas declaradas conservan el significado; el índice de Pandas no es una clave.",
)
for name, table in sources.items():
    result = ef.diff(table, original_table, keys=["row_id"], policy=policy,
                     work_dir=work / ("format-" + name))
    ef.assert_equivalent(result)
    print(name, table.num_rows, result.state, "completa:", result.complete)


CSV 10000 equivalent completa: True
JSONL 10000 equivalent completa: True
Parquet 10000 equivalent completa: True


## Contrato de la agregación

Se suman importes firmados en peniques por país y moneda, y se conserva el
número de líneas. No se convierten monedas, se deduplican transacciones ni
se eliminan devoluciones. Las sumas usan enteros exactos de 64 bits; la
normalización previa verifica su escala desde los precios decimales.
Las fechas conservan su carácter local sin zona.


In [4]:
def column(name, dtype, *, nullable=False, unit=None):
    temporal = None
    if isinstance(dtype, ef.TimestampType):
        temporal = ef.TimestampPolicy(
            granularity="us", timezone=None, naive="preserve",
            comparison="wall_time", ambiguous="reject", closed="both",
        )
    return ef.ColumnSpec(
        name=name, logical_type=dtype, nullable=nullable,
        null_policy=ef.NullPolicy(representations=("null",)),
        numeric_policy=ef.NumericPolicy(comparison="exact", unit=unit) if unit else None,
        temporal_policy=temporal,
    )

text = ef.StringType()
integer = ef.IntegerType(bits=64, signed=True)
input_columns = (
    column("row_id", integer, unit="source_position"), column("invoice_no", text),
    column("stock_code", text), column("description", text, nullable=True),
    column("quantity", integer, unit="items"),
    column("invoice_date", ef.TimestampType(unit="us", timezone=None)),
    column("unit_price", ef.DecimalType(precision=18, scale=2), unit="GBP"),
    column("customer_id", text, nullable=True), column("country", text),
    column("currency", text), column("amount_minor", integer, unit="GBP_pence"),
)
output_columns = (
    column("country", text), column("currency", text),
    column("total", integer, unit="GBP_pence"), column("rows", integer, unit="rows"),
)
def table_spec(name, columns):
    return ef.TableSpec(name=name, schema_spec=ef.TableSchema(
        columns=columns, keys=(), multiplicity="preserve", order=ef.IgnoredOrder(),
    ))
contract = ef.Contract(
    schema_version="1.0", contract_id="tutorial.kaggle.country_totals", revision=1,
    family="aggregations", inputs=(table_spec("source", input_columns),),
    output=table_spec("result", output_columns), rules=(), preconditions=(),
)
print("Contrato:", contract.contract_id, "revisión:", contract.revision)


Contrato: tutorial.kaggle.country_totals revisión: 1


## Transformaciones nativas y resultado esperado separado

Las funciones y el SQL siguientes forman parte de este cuaderno y son
confiables. No se ejecuta código procedente de los archivos descargados.
Un bucle de Python sobre los precios originales con Decimal construye la
referencia por separado; Pandas no se considera correcto por ser el primer motor.


In [5]:
registry = ef.TransformationRegistry()

def pandas_sum(tables, context):
    return tables["source"].groupby(["country", "currency"], as_index=False, dropna=False).agg(
        total=("amount_minor", "sum"), rows=("row_id", "size"),
    )

def polars_sum(tables, context):
    return tables["source"].group_by("country", "currency").agg(
        pl.col("amount_minor").sum().alias("total"), pl.len().cast(pl.Int64).alias("rows"),
    ).select("country", "currency", "total", "rows")

registry.register(ef.PandasTransformation("country_sum", pandas_sum, "tutorial.1"))
registry.register(ef.PolarsTransformation("country_sum", polars_sum, "tutorial.1"))
registry.register(ef.DuckDBTransformation(
    "country_sum",
    "SELECT country,currency,CAST(SUM(amount_minor) AS BIGINT) AS total,"
    "CAST(COUNT(*) AS BIGINT) AS rows FROM source GROUP BY country,currency",
    "tutorial.1",
))

expected_values = {}
for row in original_rows:
    key = (row["Country"], "GBP")
    total, count = expected_values.get(key, (Decimal(0), 0))
    expected_values[key] = (total + Decimal(row["UnitPrice"]) * int(row["Quantity"]), count + 1)
expected = pa.Table.from_pylist([
    {"country": key[0], "currency": key[1], "total": int(total * 100), "rows": count}
    for key, (total, count) in sorted(expected_values.items())
], schema=pa.schema([
    ("country", pa.string()), ("currency", pa.string()), ("total", pa.int64()), ("rows", pa.int64()),
]))
if sum(expected.column("rows").to_pylist()) != N:
    raise RuntimeError("El esperado no conserva las líneas seleccionadas.")

executions = {}
inputs = {"pandas": csv_table, "polars": jsonl_table, "duckdb": parquet_table}
for engine, source in inputs.items():
    execution = ef.execute(
        ef.ExecutionPlan(contract=contract, engine=engine, transformation_id="country_sum"),
        source=ef.LocalDataSource({"source": ef.ArrowInput(source)}),
        registry=registry, work_dir=work / ("execution-" + engine),
    )
    if execution.state != "completed":
        raise RuntimeError(engine + ": la ejecución no finalizó.")
    comparison = ef.compare(execution, ef.ArrowInput(expected), contract=contract,
                            policy=policy, reference="expected", work_dir=work / ("expected-" + engine))
    ef.assert_equivalent(comparison)
    executions[engine] = execution
    print(engine, "entrada:", source.num_rows, "esperado:", comparison.state)

from itertools import combinations
for left, right in combinations(executions, 2):
    comparison = ef.compare(executions[left], executions[right], contract=contract,
                            policy=policy, reference="peer", work_dir=work / (left + "-" + right))
    ef.assert_equivalent(comparison)
    print(left + "–" + right, comparison.state, "completa:", comparison.complete)


pandas entrada: 10000 esperado: equivalent
polars entrada: 10000 esperado: equivalent
duckdb entrada: 10000 esperado: equivalent
pandas–polars equivalent completa: True
pandas–duckdb equivalent completa: True
polars–duckdb equivalent completa: True


## Variables agrupadas de una ejecución

`GroupedProfileFeature` compara la salida con una entrada capturada de esa
misma ejecución. La referencia de evaluación construida arriba no se pasa al
perfil. Se utiliza explícitamente la especificación 1.1; la 1.0 conserva su
significado histórico. Un cambio exacto también puede comprobarse mediante
una regla con la misma información: estas variables no prueban por sí solas
que un modelo de aprendizaje mejore la detección.


In [6]:
metrics = (
    "prop_grupos_suma_modificada", "max_diferencia_relativa_grupo",
    "prop_grupos_cambio_desde_cero", "cobertura_grupos",
)
specification = ef.ProfileSpec(
    family="aggregations", feature_version="1.1",
    features=tuple(ef.GroupedProfileFeature(
        name=metric, metric=metric, table="result", column="total",
        reference_table="source", reference_column="amount_minor", by=("country", "currency"),
    ) for metric in metrics),
)
for engine, execution in executions.items():
    profile = ef.profile_execution(execution, spec=specification,
                                   work_dir=work / ("profile-" + engine))
    if not profile.complete:
        raise RuntimeError("La extracción del perfil no fue completa.")
    observed = {o.name: {"estado": o.state, "valor": o.value} for o in profile.vector.observations}
    if observed["prop_grupos_suma_modificada"]["valor"] != 0 or observed["cobertura_grupos"]["valor"] != 1:
        raise RuntimeError("La agregación no conservó sus grupos e importes.")
    print(engine, json.dumps(observed, ensure_ascii=False))


pandas {"prop_grupos_suma_modificada": {"estado": "available", "valor": 0.0}, "max_diferencia_relativa_grupo": {"estado": "available", "valor": 0.0}, "prop_grupos_cambio_desde_cero": {"estado": "undefined", "valor": null}, "cobertura_grupos": {"estado": "available", "valor": 1.0}}
polars {"prop_grupos_suma_modificada": {"estado": "available", "valor": 0.0}, "max_diferencia_relativa_grupo": {"estado": "available", "valor": 0.0}, "prop_grupos_cambio_desde_cero": {"estado": "undefined", "valor": null}, "cobertura_grupos": {"estado": "available", "valor": 1.0}}
duckdb {"prop_grupos_suma_modificada": {"estado": "available", "valor": 0.0}, "max_diferencia_relativa_grupo": {"estado": "available", "valor": 0.0}, "prop_grupos_cambio_desde_cero": {"estado": "undefined", "valor": null}, "cobertura_grupos": {"estado": "available", "valor": 1.0}}


## Control negativo que conserva el total global

Se trasladan 100 peniques de un país a otro en una copia del resultado.
El total conjunto se mantiene, pero cambian dos grupos. Es una modificación
controlada para comprobar el comparador, no un hallazgo sobre los datos reales.


In [7]:
changed = expected.to_pandas()
changed.loc[0, "total"] = int(changed.loc[0, "total"]) + 100
changed.loc[1, "total"] = int(changed.loc[1, "total"]) - 100
if int(changed["total"].sum()) != sum(expected.column("total").to_pylist()):
    raise RuntimeError("El control debe conservar el total global.")
grouped = ef.compare_grouped(changed, expected, by=["country", "currency"], column="total")
if not grouped.complete or grouped.modified_groups != 2:
    raise RuntimeError("El control debe modificar exactamente dos grupos.")
print("Grupos modificados:", grouped.modified_groups,
      "proporción exacta:", str(grouped.modified_group_proportion),
      "cobertura:", str(grouped.group_coverage))
negative = ef.diff(changed, expected, keys=["country", "currency"], policy=policy,
                   work_dir=work / "negative-redistribution")
if negative.state != "different" or not negative.complete:
    raise RuntimeError("La redistribución controlada debe detectarse.")
try:
    ef.assert_equivalent(negative)
except ef.EquivalenceDifferentError:
    print("Redistribución controlada: aceptación rechazada.")
else:
    raise RuntimeError("Se aceptó una redistribución incorrecta.")
summary = {"records": N, "formats_verified": 3, "native_engines": 3,
           "expected_checks": 3, "engine_pairs": 3, "grouped_profiles": 3,
           "negative_case": negative.state}
print(json.dumps(summary, ensure_ascii=False))


Grupos modificados: 2 proporción exacta: 2/15 cobertura: 1
Redistribución controlada: aceptación rechazada.
{"records": 10000, "formats_verified": 3, "native_engines": 3, "expected_checks": 3, "engine_pairs": 3, "grouped_profiles": 3, "negative_case": "different"}


## Interpretación y conservación

`equivalent` junto con una evaluación completa acredita el cumplimiento del
contrato sobre las entradas examinadas. `different` conserva una divergencia;
`not_evaluable` y una evaluación incompleta no autorizan aceptar los datos.
`assert_equivalent` aplica esta condición de aceptación sin sustituirla por
la ausencia de mensajes de error.

Los resultados de este cuaderno documentan el uso de la biblioteca. No son una
estimación de la eficacia de Isolation Forest ni resultados del corpus SERCOP.
Los datos se descargan al ejecutar y no se incluyen en el repositorio del software.
Las salidas publicadas muestran únicamente resúmenes técnicos, sin registros de clientes.

**Referencias**

Chen, D. (2015). *Online Retail* [Conjunto de datos]. UCI Machine Learning Repository.
https://doi.org/10.24432/C5BW33

Troya, H. (2026). *EquiFrame* (Versión 0.14.0) [Software]. GitHub.
https://github.com/htroya/equiframe/releases/tag/v0.14.0


In [8]:
temporary.cleanup()
print("Datos y capturas temporales eliminados.")


Datos y capturas temporales eliminados.
